#### Initialize Context & Seed Ledger

In [1]:
import sys
from pathlib import Path
from datetime import datetime
import importlib

HERE = Path.cwd()
PARENT = HERE.parent.parent.parent  # server/scripts
if str(PARENT) not in sys.path:
    sys.path.insert(0, str(PARENT))

SEED_LEDGER = Path("map/seed_ledger.csv")
if not SEED_LEDGER.parent.exists():
    SEED_LEDGER.parent.mkdir(parents=True, exist_ok=True)

In [2]:
from server.scripts.adaptive_hexsearch.h3.h3_load import load_boundary_from_json
from server.scripts.adaptive_hexsearch.h3.h3_seedtiles import init_h3
from server.scripts.adaptive_hexsearch.h3.h3_visualizemap import visualize_seed
from server.scripts.adaptive_hexsearch.h3.h3_load import load_geodf_from_csv

inner_union = load_boundary_from_json()

if not SEED_LEDGER.exists():
    seed_geodf = init_h3(inner_union)
    seed_geodf.to_csv(SEED_LEDGER, index=False)
    visualize_seed(seed_geodf, inner_union, output_path=SEED_LEDGER.with_suffix(".html"))
else:
    display("Loading existing seed ledger")
    seed_geodf = load_geodf_from_csv(SEED_LEDGER)

display(seed_geodf.head(2))

'Loading existing seed ledger'

,tile_id,seed_index,tile_path_id,h3_res,level,center_lat,center_lon,tile_size_m,geometry,children,checked,fetch_success,current
0,88194ad305fffff,0,0,8,0,51.514783,-0.090079,531.415,"POLYGON ((-0.08885 51.51926, -0.09556 51.51807...",0,False,False,False
1,88194ad32bfffff,1,1,8,0,51.509118,-0.098012,531.415,"POLYGON ((-0.09679 51.51359, -0.10349 51.5124,...",0,False,False,False


#### Select Cells

In [3]:
SELECTED_CELLS = [166, 92]
# importlib.reload(visualize_seed)
from server.scripts.adaptive_hexsearch.h3.h3_selecttiles import select_tiles_in_radius

seed_geodf["current"] = False
# seed_select = select_tiles_in_radius(seed_geodf, 51.469843, -0.138641, 800)
# selected_ids = seed_select['tile_id'].tolist()
# seed_geodf["current"] = seed_geodf['tile_id'].apply(lambda x: x in selected_ids)

seed_geodf["current"] = seed_geodf['tile_path_id'].apply(lambda x: x in SELECTED_CELLS)

display("Updating Ledger")
seed_geodf.to_csv(SEED_LEDGER, index=False)
visualize_seed(seed_geodf, inner_union, output_path=SEED_LEDGER.with_suffix(".html"))

'Updating Ledger'

Saved seed H3 map to: map\seed_ledger.html


#### Run APIs

In [4]:
ENABLE_API = True
from server.scripts.adaptive_hexsearch.h3.h3_subdivision import run_h3_recursive_division

seed_select = seed_geodf[seed_geodf['current'] == True]
cell_geodf = await run_h3_recursive_division(seed_select, inner_union, disable_api=not ENABLE_API)

API calls executed for 92: 1 | failures: 1
API calls executed for 166: 2 | failures: 2
Run complete | Final Cell Count: 2 | Stats: {'api_calls': 2, 'discarded': 0, 'added': 0, 'api_failures': 2}


File API Response and Update Ledger

In [5]:
from server.scripts.adaptive_hexsearch.h3.h3_visualizemap import visualize_divisions

response_valid = cell_geodf is not None and len(cell_geodf) > 0 \
    and 'tile_id' in cell_geodf.columns and 'fetch_success' in cell_geodf.columns

if ENABLE_API and response_valid:
    display("Fetch Complete.")
    OUT_LEDGER = SEED_LEDGER.parent / "batch" / f"{datetime.now().strftime('%Y-%m-%d')}-{len(seed_select)}-{len(cell_geodf)}.csv"
    if not OUT_LEDGER.parent.exists(): OUT_LEDGER.parent.mkdir(parents=True, exist_ok=True)
    cell_geodf.to_csv(OUT_LEDGER, index=False)
    visualize_divisions(cell_geodf, inner_union, output_path=OUT_LEDGER.with_suffix(".html"))

    success_ids = cell_geodf.loc[cell_geodf['fetch_success'] == True, 'tile_id'].dropna()
    seed_geodf.loc[seed_geodf['tile_id'].isin(success_ids), 'fetch_success'] = True
else:
    display("Response InValid")

display("Updating Ledger")
seed_geodf.loc[seed_geodf['current'] == True, 'checked'] = True
seed_geodf['current'] = False
seed_geodf.to_csv(SEED_LEDGER, index=False)
visualize_seed(seed_geodf, inner_union, output_path=SEED_LEDGER.with_suffix(".html"))

'Fetch Complete.'

Saved mock adaptive H3 map to: map\batch\2026-06-07-2-2.html


'Updating Ledger'

Saved seed H3 map to: map\seed_ledger.html


In [6]:
cell_geodf

,tile_id,seed_index,tile_path_id,h3_res,level,center_lat,center_lon,tile_size_m,geometry,children,checked,fetch_success,current,result_count
0,88194ad16bfffff,92,92,8,0,51.488532,-0.141927,531.415,"POLYGON ((-0.1407 51.49301, -0.14741 51.49181,...",0,True,False,True,None
1,88194ad11dfffff,166,166,8,0,51.463176,-0.116967,531.415,"POLYGON ((-0.11574 51.46765, -0.12245 51.46646...",0,True,False,True,None


#### Update Division Map

In [7]:
import geopandas as gpd
import pandas as pd

DIVISION_LEDGER = SEED_LEDGER.with_name("division_ledger.csv")
division_geodf = load_geodf_from_csv(DIVISION_LEDGER)
combined = pd.concat([division_geodf, cell_geodf], ignore_index=True)
combined.drop_duplicates(subset=['tile_id', 'tile_path_id'], keep='last', inplace=True)
# combined.dropna(subset=['seed_index'], inplace=True)
division_geodf = gpd.GeoDataFrame(combined, geometry="geometry", crs=division_geodf.crs)

# division_geodf.to_csv(DIVISION_LEDGER, index=False)
visualize_divisions(division_geodf, inner_union, output_path=DIVISION_LEDGER.with_suffix(".html"))

Saved mock adaptive H3 map to: map\division_ledger.html


In [8]:
division_geodf=division_geodf[division_geodf['tile_path_id']=="264"]
division_geodf

,tile_id,tile_path_id,h3_res,level,center_lat,center_lon,tile_size_m,geometry,children,checked,fetch_success,result_count,seed_index,current
